# Orphaned Metastore Entry Does Not Abort Its Database — Behavior Test (WF-343)

End-to-end verification of the WF-343 fix in `migration_dag_mapr_to_s3.py`, run against the shared
MapR edge node in the `hadoop-ssh` namespace. Triggers a real `source_to_s3_migration` DAG run and
asserts on the Airflow task states, the Iceberg tracking tables, and the HTML report.

### What changed

A Hive table whose `LOCATION` directory has been deleted (data purged, metastore entry left behind)
made `spark.table()` raise during discovery. The remote script labelled it `FAILED`, and the driver
gate treated any non-`(TABLE_NOT_FOUND, DATABASE_NOT_FOUND)` error as fatal — so **one orphaned
table aborted discovery for its entire database**, and every table in it went unmigrated.

The fix emits `source_path_exists = bool(fs.exists(root))` from the remote script — a value it
already computed and discarded — and keys the decision on that instead of on the exception text.
Wording cannot distinguish "the table root is gone" from "a leaf file under it is gone", because
Spark's file index names leaf files; `fs.exists()` on the root can. A verifiably-absent root becomes
`SOURCE_PATH_NOT_FOUND`, which is skipped like `TABLE_NOT_FOUND` already was. `None` (the probe
could not answer) still aborts.

### What this notebook verifies

| Step | Assertion | Why it matters |
|---|---|---|
| 2 | `fs.exists()` is `False` for the orphans and `True` for the healthy tables | The ground truth the whole fix rests on, measured on real HDFS rather than mocked |
| 8 | The mapped `discover_tables_via_spark_ssh` instance is `success` | The WF-343 regression itself: this task used to fail and take the database with it |
| 9 | Both orphans land as `SOURCE_PATH_NOT_FOUND` | The new status is reached and recorded |
| 9 | `legacy_events` (TEXTFILE) is **not** `EMPTY_SOURCE` | A non-Parquet orphan never raises, so it used to pass as a *successful* empty copy |
| 9 | `orders` and `customers` reach `VALIDATED` | The healthy siblings still migrate — the whole point of the ticket |
| 9 | `error_message` names the missing path | Ops needs the path to decide between dropping the entry and restoring data |
| 10 | Run status is `COMPLETED_WITH_MISSING`, not `COMPLETED` | A skipped table must never leave the run looking clean |
| 11 | Report shows a `SOURCE PATH MISSING` card and badge | The condition is visible without reading task logs |

### Fixture

One database, `wf343_db`, holding two healthy tables and two orphans, discovered by a **single**
Excel row (`table = *`). That matters: all four tables share one mapped discovery task, so before
the fix the two healthy tables were collateral damage of the two orphans. One Excel row is the
tightest reproduction of the ticket.

| Table | Format | State | Expected outcome |
|---|---|---|---|
| `orders` | Parquet, partitioned by `dt` | intact | `VALIDATED` |
| `customers` | Parquet | intact | `VALIDATED` |
| `phone_pref_ref` | Parquet | `LOCATION` deleted | `SOURCE_PATH_NOT_FOUND` (raises during discovery) |
| `legacy_events` | TEXTFILE | `LOCATION` deleted | `SOURCE_PATH_NOT_FOUND` (does *not* raise) |

### Prerequisites

- `paramiko` reachable from JupyterHub, and SSH credentials for the edge pod.
- Airflow webserver reachable, with permission to write Variables and trigger DAGs.
- JH Spark session with S3 write access to the tenant bucket, and read access to the tracking DB.
- The branch under test (`fix/wf-343-discovery-source-path-not-found`) deployed to the Airflow DAGs
  bucket. Against `main` this notebook is expected to **fail at Step 8** — which is the demonstration.

Reference: [MapR Edge Node Setup Guide](https://www.notion.so/MapR-Edge-Node-Setup-Guide-351984e820aa80e29cd9fa0d7fd127d9).

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────────────
import os

# Nothing environment-specific is defaulted: this repo is public, so hosts, buckets,
# namespaces and credentials come from the environment. The check at the bottom names
# whatever is missing, rather than letting it surface as an SSH or S3 error later.

# --- Edge node (SSH) ---
# Leave EDGE_SSH_HOST unset and Step 0 adopts the host from the DAG's own SSH connection, which
# is the only value that can be right: the fixture has to be seeded on the node discovery reads.
# It is *not* the shared edge node in its own namespace — that one is a different metastore.
SSH_CONN_ID  = os.environ.get("CLUSTER_SSH_CONN_ID", "cluster_edge_ssh")
SSH_HOST     = os.environ.get("EDGE_SSH_HOST", "")
SSH_PORT     = int(os.environ.get("EDGE_SSH_PORT", "22"))
SSH_USER     = os.environ.get("EDGE_SSH_USER", "root")
SSH_PASSWORD = os.environ.get("EDGE_SSH_PASSWORD", "")
MAPR_USER       = "root"
MAPR_TICKETFILE = "/tmp/maprticket"

# --- Airflow REST API ---
# In-cluster URL bypasses Keycloak, which only fronts the public Ingress. The webserver
# behind it uses the FAB auth manager, so the token comes from /auth/token.
# AIRFLOW_BASE_URL example: "http://airflow-api-server.<tenant-namespace>.svc.cluster.local:8080"
AIRFLOW_BASE_URL = os.environ.get("AIRFLOW_BASE_URL", "")
AIRFLOW_USERNAME = os.environ.get("AIRFLOW_USERNAME", "")
AIRFLOW_PASSWORD = os.environ.get("AIRFLOW_PASSWORD", "")
DAG_ID           = "source_to_s3_migration"

# --- S3 ---
# S3_TENANT_PREFIX example: f"s3a://{BUCKET}/<tenant>"
BUCKET        = os.environ.get("S3_BUCKET", "")
TENANT_PREFIX = os.environ.get("S3_TENANT_PREFIX", "")
EXCEL_S3_PATH = f"{TENANT_PREFIX}/configs/wf343_orphan_test.xlsx"

# --- Tracking + report ---
# Step 4 pins all three of these as Airflow Variables, so whatever is set here is what the
# DAG uses. Do not set only the locations: `tracking_database` falls back to a hardcoded
# 'migration_tracking' when its Variable is unset, which silently writes a suffixed
# deployment's rows into the shared database while the locations point somewhere else.
TRACKING_DB       = os.environ.get("MIGRATION_TRACKING_DB", "migration_tracking")
TRACKING_LOCATION = f"{TENANT_PREFIX}/{TRACKING_DB}"
REPORT_LOCATION   = os.environ.get("MIGRATION_REPORT_LOCATION", f"{TENANT_PREFIX}/migration_reports")

# --- Fixture on the edge node ---
SRC_DB        = "wf343_db"
DEST_DB       = "wf343_db_copy"
HDFS_SRC_BASE = "/user/root/wf343_src"

# Tables whose LOCATION is deleted after seeding, leaving the metastore entry behind.
# phone_pref_ref is Parquet, so spark.table() lists files and raises.
# legacy_events is TEXTFILE, so its schema comes from the catalog and nothing raises — that
# table used to reach DistCp with 0 files and be reported EMPTY_SOURCE, a false success.
ORPHAN_TABLES  = ["legacy_events", "phone_pref_ref"]
HEALTHY_TABLES = ["customers", "orders"]
ALL_TABLES     = sorted(ORPHAN_TABLES + HEALTHY_TABLES)

# The pseudo-distributed edge cluster cannot launch 50 YARN containers.
DISTCP_MAPPERS = "1"

# EDGE_SSH_HOST is absent on purpose: Step 0 resolves it from the DAG's SSH connection.
REQUIRED_ENV = {
    "EDGE_SSH_PASSWORD": SSH_PASSWORD,
    "AIRFLOW_BASE_URL": AIRFLOW_BASE_URL,
    "AIRFLOW_USERNAME": AIRFLOW_USERNAME,
    "AIRFLOW_PASSWORD": AIRFLOW_PASSWORD,
    "S3_BUCKET": BUCKET,
    "S3_TENANT_PREFIX": TENANT_PREFIX,
}

print("Configuration loaded.")
print(f"  SSH target      : {SSH_USER}@{SSH_HOST or '(from ' + SSH_CONN_ID + ')'}:{SSH_PORT}")
print(f"  Airflow URL     : {AIRFLOW_BASE_URL or '(unset)'}")
print(f"  Airflow user    : {AIRFLOW_USERNAME or '(unset)'}")
print(f"  S3 bucket       : {BUCKET or '(unset)'}")
print(f"  Tenant prefix   : {TENANT_PREFIX or '(unset)'}")
print(f"  Tracking DB     : {TRACKING_DB}  at {TRACKING_LOCATION}")
print(f"  Source database : {SRC_DB}  (HDFS base {HDFS_SRC_BASE})")
print(f"  Healthy tables  : {HEALTHY_TABLES}")
print(f"  Orphan tables   : {ORPHAN_TABLES}")
print(f"  Excel S3 path   : {EXCEL_S3_PATH}")

_unset = sorted(name for name, value in REQUIRED_ENV.items() if not value)
if _unset:
    raise RuntimeError(
        "Set these environment variables before running the rest of the notebook: "
        + ", ".join(_unset)
    )
print("\nAll required environment variables are set.")

In [ ]:
# ── Helpers ─────────────────────────────────────────────────────────
import re
import subprocess
import sys

import requests

try:
    import paramiko
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "paramiko"])
    import paramiko

from py4j.java_gateway import java_import
from pyspark.sql import SparkSession

try:
    _ = spark
    print(f"Using existing Spark session (version {spark.version})")
except NameError:
    spark = SparkSession.builder \
        .appName("wf343-orphan-e2e") \
        .enableHiveSupport() \
        .getOrCreate()
    print(f"Created Spark session (version {spark.version})")

java_import(spark._jvm, "org.apache.hadoop.fs.*")

# Login-shell init scripts on this pod emit these on every SSH session. Not errors.
_LOGIN_NOISE = re.compile(
    r"^(mesg: ttyname failed: Inappropriate ioctl for device"
    r"|ls: cannot access '/opt/spark/lib/spark-assembly-\*\.jar': No such file or directory)$"
)
# pyspark on the edge node interleaves log4j output with our prints.
_SPARK_LOG = re.compile(
    r"^([0-9]{2}/[0-9]{2}/[0-9]{2} |INFO |WARN |ERROR |Setting default log level"
    r"|To adjust logging|Welcome to|Using Python|SparkSession available|Python version)"
)


def _strip_login_noise(text):
    return "\n".join(line for line in text.splitlines() if not _LOGIN_NOISE.match(line))


def _ssh_connect():
    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    client.connect(
        hostname=SSH_HOST, port=SSH_PORT,
        username=SSH_USER, password=SSH_PASSWORD,
        timeout=30, allow_agent=False, look_for_keys=False,
    )
    return client


def pod_exec(script, check=True, timeout=900, quiet=False):
    """Run a bash script in the edge pod via SSH login shell. Returns stdout."""
    if not quiet:
        preview = script if len(script) <= 200 else script[:197] + "..."
        print(f"$ ({SSH_HOST}) {preview}")
    client = _ssh_connect()
    try:
        stdin, stdout, stderr = client.exec_command("bash -l", timeout=timeout)
        stdin.write(script + "\n")
        stdin.channel.shutdown_write()
        out = stdout.read().decode()
        err = stderr.read().decode()
        rc = stdout.channel.recv_exit_status()
    finally:
        client.close()
    out = _strip_login_noise(out)
    err = _strip_login_noise(err)
    if not quiet:
        if out.strip():
            print(out.rstrip())
        if err.strip():
            print(err.rstrip())
    if check and rc != 0:
        # Surface the remote output even when quiet: the exit code on its own says
        # nothing about what went wrong on the far end.
        if quiet:
            for stream in (out, err):
                if stream.strip():
                    print(stream.rstrip())
        raise RuntimeError(f"pod exec exited {rc}")
    return out


def edge_pyspark(script, label, timeout=900, echo=True):
    """Run a PySpark script on the edge node the way the DAG runs its own discovery: write it
    to /tmp and pipe it into `pyspark --master local[*]`. Returns stdout minus log4j chatter.

    The remote scripts must be **Python 2 compatible** — no f-strings. The edge node dates
    from Hadoop 2.7.7 and its pyspark may run Python 2.7, which is why the DAG's own 455-line
    discovery script uses `.format()` throughout and carries a coding header. Step 0b prints
    what this node actually resolves.

    Placeholders are substituted with `__NAME__` rather than interpolated, so the scripts can
    contain Spark SQL braces freely.
    """
    remote = f"/tmp/wf343_{label}.py"
    pod_exec("cat > " + remote + " << 'WF343EOF'\n" + script + "\nWF343EOF", quiet=True)
    # Trailing echo captures pyspark's own exit code: piping a script into it is
    # non-interactive, so a SyntaxError or uncaught exception does exit non-zero.
    raw = pod_exec(
        "cd /tmp && pyspark --master 'local[*]' < " + remote + " 2>&1; echo WF343_EXIT=$?",
        timeout=timeout, quiet=True, check=False,
    )
    lines = raw.splitlines()
    rc = next((int(x.split("=", 1)[1]) for x in lines if x.startswith("WF343_EXIT=")), None)
    out = "\n".join(
        x for x in lines if not _SPARK_LOG.match(x) and not x.startswith("WF343_EXIT=")
    )
    if echo or rc:
        print(out.rstrip())
    if rc:
        raise RuntimeError(f"{label} script exited {rc} on the edge node — see output above")
    return out


_AIRFLOW_TOKEN = {"value": None}


def _airflow_token():
    if _AIRFLOW_TOKEN["value"]:
        return _AIRFLOW_TOKEN["value"]
    if not AIRFLOW_USERNAME or not AIRFLOW_PASSWORD:
        raise RuntimeError("Set AIRFLOW_USERNAME and AIRFLOW_PASSWORD env vars first.")
    url = AIRFLOW_BASE_URL.rstrip("/") + "/auth/token"
    resp = requests.post(url, json={"username": AIRFLOW_USERNAME, "password": AIRFLOW_PASSWORD},
                         timeout=30)
    if not resp.ok:
        raise RuntimeError(f"POST {url} -> {resp.status_code}: {resp.text[:300]}")
    _AIRFLOW_TOKEN["value"] = resp.json()["access_token"]
    return _AIRFLOW_TOKEN["value"]


def airflow_request(method, path, **kwargs):
    headers = {"Authorization": f"Bearer {_airflow_token()}", **kwargs.pop("headers", {})}
    url = AIRFLOW_BASE_URL.rstrip("/") + path
    resp = requests.request(method, url, headers=headers, timeout=30, **kwargs)
    if resp.status_code == 401:
        _AIRFLOW_TOKEN["value"] = None
        headers["Authorization"] = f"Bearer {_airflow_token()}"
        resp = requests.request(method, url, headers=headers, timeout=30, **kwargs)
    if not resp.ok:
        raise RuntimeError(f"{method} {url} -> {resp.status_code}: {resp.text[:500]}")
    return resp.json() if resp.content else {}


def _fs(path):
    return spark._jvm.org.apache.hadoop.fs.FileSystem.get(
        spark._jvm.java.net.URI(path), spark._jsc.hadoopConfiguration()
    )


def s3_delete(path):
    fs = _fs(path)
    p = spark._jvm.org.apache.hadoop.fs.Path(path)
    if fs.exists(p):
        fs.delete(p, True)
        print(f"  Deleted: {path}")
    else:
        print(f"  Not found (skip): {path}")


def try_drop_database(db, why):
    """Drop a Hive database, tolerating a Ranger denial.

    The destination database belongs to the Airflow service account, which is what creates
    it; this JH user usually has no `drop` privilege on it. Dropping from here is tidying,
    not a prerequisite, so a denial is reported as expected rather than as an error.
    """
    try:
        spark.sql(f"DROP DATABASE IF EXISTS {db} CASCADE")
        print(f"  dropped database {db}")
        return True
    except Exception as e:
        msg = str(e)
        if "AccessControlException" in msg or "Permission denied" in msg:
            print(f"  {db}: no drop privilege for this user — expected, {why}")
        else:
            print(f"  {db}: drop failed — {msg.splitlines()[0][:160]}")
        return False


print("Helpers ready.")

---
## Step 0 — Agree on the edge node, then verify it is reachable

The fixture must be seeded on the node the DAG reads, and the DAG reads whichever host its
`cluster_edge_ssh` connection names. Pointing the notebook at a different edge node — the shared one
in its own namespace, say — puts `wf343_db` in a metastore discovery never queries, and every run
then reports `DATABASE_NOT_FOUND` for the `*` token while the fixture check passes. So take the host
from the connection rather than restating it here.

Then the usual liveness check: SSH proves port 22 is up, and the Hive metastore (9083) and
HiveServer2 (10000) should both be listening. If they are missing the pod was probably just
restarted — wait ~90s and re-run.

In [ ]:
try:
    _conn = airflow_request("GET", f"/api/v2/connections/{SSH_CONN_ID}")
    dag_ssh_host = (_conn.get("host") or "").strip()
    print(f"  {SSH_CONN_ID} host : {dag_ssh_host or '(no host set)'}")
except RuntimeError as e:
    dag_ssh_host = ""
    print(f"  could not read the {SSH_CONN_ID} connection: {e}")
    print("  falling back to EDGE_SSH_HOST — check it names the same node the DAG uses")

if dag_ssh_host and not SSH_HOST:
    SSH_HOST = dag_ssh_host
    print(f"  EDGE_SSH_HOST unset, adopting the DAG's host: {SSH_HOST}")
elif dag_ssh_host and dag_ssh_host != SSH_HOST:
    raise RuntimeError(
        f"Edge node mismatch. EDGE_SSH_HOST is {SSH_HOST!r} but the DAG's {SSH_CONN_ID} connection "
        f"points at {dag_ssh_host!r}. Seeding one node while discovery reads the other makes every "
        f"run report DATABASE_NOT_FOUND. Unset EDGE_SSH_HOST to adopt the DAG's host."
    )

assert SSH_HOST, (
    f"No edge host: EDGE_SSH_HOST is unset and the {SSH_CONN_ID} connection could not be read."
)
print(f"\nUsing edge node: {SSH_USER}@{SSH_HOST}:{SSH_PORT}\n")

pod_exec("hostname && uptime")
pod_exec("ss -tlnp 2>/dev/null | grep -E ':(22|9083|10000) ' || netstat -tlnp 2>/dev/null | grep -E ':(22|9083|10000) '")

---
## Step 0b — What Python does the edge node's pyspark run?

Everything this notebook ships to the edge node has to run under that interpreter. The DAG's own
discovery script is 455 lines with no f-strings and a `# -*- coding: utf-8 -*-` header, so Python
2.7 is a live possibility on a Hadoop 2.7.7-era node. The remote scripts below use `.format()` for
that reason; this cell records what is actually there.

In [ ]:
print(pod_exec(
    "echo PYSPARK_PYTHON=${PYSPARK_PYTHON:-'(unset)'}; "
    "python -c 'import sys; print(\"python=\" + sys.version.split()[0])' 2>&1; "
    "spark-submit --version 2>&1 | grep -iE 'version [0-9]' | head -2",
    quiet=True,
))

---
## Step 1 — Seed the source database

Seeded through `pyspark --master local[*]` on the edge node rather than beeline, so the tables land
in exactly the metastore and warehouse the DAG's own discovery script will read.

All four tables are `EXTERNAL` with an explicit `LOCATION` under `HDFS_SRC_BASE`, which is what a
real datalake table looks like and makes orphaning a single `hdfs dfs -rm -r` away. The cell drops
and recreates the database, so it is safe to re-run.

`legacy_events` is deliberately `TEXTFILE`. A Parquet orphan raises when Spark lists its files; a
TEXTFILE orphan does not, because its schema comes from the catalog and the only throwing call
(`SELECT COUNT(*)`) is swallowed remotely. The two formats exercise the two different code paths
that have to reach the same verdict.

In [ ]:
SEED_SCRIPT = r"""# -*- coding: utf-8 -*-
# No f-strings: this runs under the edge node's pyspark interpreter, which may be Python 2.7.
# Same constraint the DAG's own remote discovery script works under.
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("wf343-seed").enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel("WARN")

SRC_DB = "__SRC_DB__"
BASE = "__BASE__"

spark.sql("DROP DATABASE IF EXISTS {0} CASCADE".format(SRC_DB))
spark.sql("CREATE DATABASE {0}".format(SRC_DB))
print("Recreated database " + SRC_DB)

# healthy, partitioned Parquet
spark.sql(
    "CREATE EXTERNAL TABLE {0}.orders "
    "(order_id INT, customer_id INT, amount DOUBLE) "
    "PARTITIONED BY (dt STRING) STORED AS PARQUET "
    "LOCATION '{1}/orders'".format(SRC_DB, BASE)
)
spark.sql(
    "INSERT INTO {0}.orders PARTITION (dt='2026-01-01') "
    "VALUES (1, 10, 99.5), (2, 11, 12.0)".format(SRC_DB)
)
spark.sql(
    "INSERT INTO {0}.orders PARTITION (dt='2026-01-02') "
    "VALUES (3, 12, 45.25)".format(SRC_DB)
)

# healthy, non-partitioned Parquet
spark.sql(
    "CREATE EXTERNAL TABLE {0}.customers "
    "(customer_id INT, name STRING, city STRING) "
    "STORED AS PARQUET LOCATION '{1}/customers'".format(SRC_DB, BASE)
)
spark.sql(
    "INSERT INTO {0}.customers "
    "VALUES (10, 'ada', 'riga'), (11, 'grace', 'london'), (12, 'alan', 'oslo')".format(SRC_DB)
)

# to be orphaned: Parquet, so spark.table() lists files and raises once the root is gone
spark.sql(
    "CREATE EXTERNAL TABLE {0}.phone_pref_ref "
    "(pref_id INT, phone STRING, opted_in BOOLEAN) "
    "STORED AS PARQUET LOCATION '{1}/phone_pref_ref'".format(SRC_DB, BASE)
)
spark.sql(
    "INSERT INTO {0}.phone_pref_ref "
    "VALUES (1, '+371000000', true), (2, '+442000000', false)".format(SRC_DB)
)

# to be orphaned: TEXTFILE, whose schema comes from the catalog, so nothing raises
spark.sql(
    "CREATE EXTERNAL TABLE {0}.legacy_events "
    "(event_id INT, kind STRING) "
    "STORED AS TEXTFILE LOCATION '{1}/legacy_events'".format(SRC_DB, BASE)
)
spark.sql(
    "INSERT INTO {0}.legacy_events VALUES (1, 'click'), (2, 'view')".format(SRC_DB)
)

for t in ["customers", "legacy_events", "orders", "phone_pref_ref"]:
    n = spark.sql("SELECT COUNT(*) c FROM {0}.{1}".format(SRC_DB, t)).collect()[0].c
    print("  seeded {0}.{1}: {2} row(s)".format(SRC_DB, t, n))
""".replace("__SRC_DB__", SRC_DB).replace("__BASE__", HDFS_SRC_BASE)

seed_out = edge_pyspark(SEED_SCRIPT, "seed")
for t in ALL_TABLES:
    assert f"seeded {SRC_DB}.{t}" in seed_out, f"seed did not report {t} — see output above"
print("\nAll four tables seeded.")

---
## Step 2 — Orphan two tables, then probe the filesystem the way discovery does

Deleting the data directory while leaving the metastore entry is the orphaned-entry condition
WF-343 is about.

The second cell then runs the **same `fs.exists()` probe the fixed discovery script runs**, against
real HDFS on the edge node. This is the one assumption the unit tests cannot check, because they
mock the filesystem: that `exists()` on a deleted table root returns `False`, and `True` for an
intact one. Everything downstream is a consequence of these two values.

In [ ]:
delete_cmds = "\n".join(
    f"hdfs dfs -rm -r -skipTrash {HDFS_SRC_BASE}/{t}" for t in ORPHAN_TABLES
)
pod_exec("set -x\n" + delete_cmds)

print()
pod_exec(f"hdfs dfs -ls {HDFS_SRC_BASE}")

In [ ]:
PROBE_SCRIPT = r"""# -*- coding: utf-8 -*-
# Python 2 compatible: no f-strings. See SEED_SCRIPT.
from py4j.java_gateway import java_import
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("wf343-probe").enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel("WARN")
java_import(spark._jvm, "org.apache.hadoop.fs.*")

SRC_DB = "__SRC_DB__"


def location_of(db, tbl):
    rows = spark.sql("DESCRIBE FORMATTED {0}.{1}".format(db, tbl)).collect()
    for row in rows:
        if (row.col_name or "").strip() == "Location":
            return (row.data_type or "").strip()
    return ""


# Mirrors the probe in discover_tables_via_spark_ssh: exists() on the table's LOCATION root.
# fs.exists() returns False only for FileNotFoundException and propagates every other
# IOException, so a permissions or connectivity fault cannot be mistaken for absence.
for tbl in sorted([t.name for t in spark.catalog.listTables(SRC_DB)]):
    loc = location_of(SRC_DB, tbl)
    exists = None
    try:
        fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
            spark._jvm.java.net.URI(loc), spark._jsc.hadoopConfiguration()
        )
        exists = bool(fs.exists(spark._jvm.org.apache.hadoop.fs.Path(loc)))
    except Exception as e:
        print("WF343_PROBE_ERROR={0}|{1}|{2}".format(tbl, loc, str(e)[:120]))
    print("WF343_PROBE={0}|{1}|{2}".format(tbl, loc, exists))
""".replace("__SRC_DB__", SRC_DB)

probe_out = edge_pyspark(PROBE_SCRIPT, "probe", echo=False)

probe = {}
for line in probe_out.splitlines():
    if line.startswith("WF343_PROBE="):
        tbl, loc, exists = line[len("WF343_PROBE="):].split("|", 2)
        probe[tbl] = {"location": loc, "exists": exists}
    elif line.startswith("WF343_PROBE_ERROR="):
        print("  probe error: " + line[len("WF343_PROBE_ERROR="):])

# A traceback in the remote script leaves the pyspark REPL exit code at 0, so an empty
# parse is the only symptom. Show the raw output rather than asserting on an empty dict.
if not probe:
    print("No WF343_PROBE lines came back. Raw remote output:\n")
    print(probe_out)
    raise AssertionError("probe script produced no results — see the output above")

print(f"  {'table':<18} {'fs.exists()':<12} location")
print(f"  {'-'*18} {'-'*12} {'-'*50}")
for tbl in sorted(probe):
    print(f"  {tbl:<18} {probe[tbl]['exists']:<12} {probe[tbl]['location']}")

assert set(probe) == set(ALL_TABLES), \
    f"metastore lists {sorted(probe)}, expected {ALL_TABLES} — seed or delete went wrong"

for t in ORPHAN_TABLES:
    assert probe[t]["exists"] == "False", (
        f"{t}: fs.exists() returned {probe[t]['exists']!r}, expected False. The fixture is not "
        f"orphaned, so nothing below tests WF-343."
    )
for t in HEALTHY_TABLES:
    assert probe[t]["exists"] == "True", \
        f"{t}: fs.exists() returned {probe[t]['exists']!r}, expected True — only the orphans should be gone"

print(f"\nPASS — {len(ORPHAN_TABLES)} root(s) verifiably absent, {len(HEALTHY_TABLES)} intact, "
      f"all {len(ALL_TABLES)} still registered in the metastore.")

---
## Step 3 — Install the MapR `maprlogin` stub and ticket file

`validate_prerequisites` runs `maprlogin print | grep -q <user>` over SSH. The pod is not a real
MapR node, so stub a binary that always reports a valid ticket for `root` and persist
`MAPR_TICKETFILE_LOCATION` so it survives across SSH sessions.

In [ ]:
MAPR_SETUP_SCRIPT = r'''
# Note: deliberately no `set -o pipefail` — `grep -q` closes the pipe on first match, which
# sends SIGPIPE (exit 141) to maprlogin. The DAG's validate_prerequisites runs without
# pipefail for the same reason.
set -eu

cat > /usr/local/bin/maprlogin << 'EOF'
#!/bin/bash
if [ "$1" = "print" ]; then
  echo "MapR credentials (UID 0) for user: root"
  echo "  created: $(date)"
  echo "  expires: $(date -d '+7 days' 2>/dev/null || date)"
  echo "  cluster: test-cluster"
  exit 0
fi
echo "maprlogin: unknown command '$1'"
exit 1
EOF
chmod +x /usr/local/bin/maprlogin

mkdir -p /tmp
cat > /tmp/maprticket << 'EOF'
MAPR_TICKET
cluster=test-cluster
user=root
uid=0
created=0
expires=9999999999
EOF

for rc in /root/.profile /root/.bashrc; do
  grep -q MAPR_TICKETFILE_LOCATION "$rc" || echo 'export MAPR_TICKETFILE_LOCATION=/tmp/maprticket' >> "$rc"
done

if maprlogin print 2>/dev/null | grep -q "root"; then
  echo "TICKET CHECK: PASSED"
else
  echo "TICKET CHECK: FAILED"
  exit 1
fi
'''

pod_exec(MAPR_SETUP_SCRIPT)

---
## Step 4 — Configure Airflow Variables

`migration_tracking_database` is set alongside the two locations, and pinning all three together is
the point. `get_config()` resolves each independently — Airflow Variable, then env var, then a
hardcoded default — so setting only the locations leaves the database on its `'migration_tracking'`
fallback. A suffixed deployment then writes its rows into the *shared* database while its files land
under the suffixed prefix, and Step 9 queries a database the run never touched.

The two locations must also be pinned to the tenant bucket: left unset the DAG falls back to a
different bucket and region, and `generate_report` fails writing the HTML with an S3 301 redirect.
`cluster_type` stays `MapR` because the edge node is Hadoop 2.7.7, and `HDP` emits an
`S3ACommitterFactory` that only exists in Hadoop 3.1+.

In [ ]:
VARIABLES = {
    "auth_method": "mapr",
    "mapr_user": MAPR_USER,
    "mapr_ticketfile_location": MAPR_TICKETFILE,
    "migration_distcp_mappers": DISTCP_MAPPERS,
    "migration_report_location": REPORT_LOCATION,
    "migration_tracking_database": TRACKING_DB,
    "migration_tracking_location": TRACKING_LOCATION,
    "cluster_type": "MapR",
}

for key, value in VARIABLES.items():
    body = {"key": key, "value": value}
    try:
        airflow_request("PATCH", f"/api/v2/variables/{key}", json=body)
        action = "updated"
    except RuntimeError as e:
        if "404" in str(e):
            airflow_request("POST", "/api/v2/variables", json=body)
            action = "created"
        else:
            raise
    print(f"  {action}: {key} = {value}")

---
## Step 5 — Build and upload the Excel migration config

**One row, `table = *`.** The wildcard expands to all four tables inside a single mapped
`discover_tables_via_spark_ssh` instance, so the healthy tables and the orphans share one task —
which is precisely why, before the fix, two orphans took two healthy tables down with them.
Splitting them across rows would isolate the failures and hide the bug.

In [ ]:
from io import BytesIO

import pandas as pd

rows = [
    {"database": SRC_DB, "table": "*", "partition_filter": "",
     "dest_database": DEST_DB, "bucket": BUCKET, "endpoint": ""},
]
df = pd.DataFrame(rows)

buf = BytesIO()
df.to_excel(buf, index=False, engine="openpyxl")
xlsx_bytes = buf.getvalue()

fs = _fs(EXCEL_S3_PATH)
p = spark._jvm.org.apache.hadoop.fs.Path(EXCEL_S3_PATH)
out = fs.create(p, True)
try:
    out.write(xlsx_bytes)
finally:
    out.close()

print(f"Uploaded {len(xlsx_bytes)} bytes to {EXCEL_S3_PATH}")
df

---
## Step 6 — Wipe the destination (pre-run reset)

DistCp runs with `-update`, so a leftover destination from a previous execution would make this run
incremental and change the file/size comparisons. Clearing the S3 prefix keeps every run a first run.

Dropping the destination *Hive* database is tidying on top of that, and Ranger will usually deny it:
the database belongs to the Airflow service account, not to your JH user. That denial is expected and
does not affect anything the notebook asserts — `create_hive_tables` reuses the existing database and
reports `table_already_existed`, and the WF-343 assertions are about discovery statuses, not about
whether the destination was created fresh.

In [ ]:
# Clearing the S3 prefix is the part that matters: it is what makes this a first run
# rather than an incremental one.
s3_delete(f"{TENANT_PREFIX}/{DEST_DB}")
try_drop_database(DEST_DB, "create_hive_tables will reuse the existing database")
print("\nDestination reset — S3 data cleared.")

---
## Step 7 — Verify the fixture, then trigger

The check and the trigger are one cell on purpose. Anything that drops `wf343_db` between them — a
stray Step 12, a re-run of Step 1 that failed after its `DROP DATABASE`, someone else on the shared
edge node — produces a run whose discovery emits a single `DATABASE_NOT_FOUND` record for the `*`
token. That run still reports `success`, so you would not find out until Step 9, ten minutes later,
from a table set of `['*']`.

So: confirm the metastore lists all four tables and that only the orphans' data directories are
missing, and POST the trigger in the same breath.

In [ ]:
from datetime import datetime, timezone

# --- verify the fixture, immediately before triggering ---
FIXTURE_CHECK = f"""
echo WF343_HDFS_START
hdfs dfs -ls {HDFS_SRC_BASE} 2>/dev/null || echo "(base path missing)"
echo WF343_TABLES_START
beeline -u jdbc:hive2://localhost:10000 -e 'SHOW TABLES IN {SRC_DB};' --silent=true 2>&1
"""

out = pod_exec(FIXTURE_CHECK, quiet=True, check=False)
hdfs_part, _, tables_part = out.partition("WF343_TABLES_START")
print(hdfs_part.replace("WF343_HDFS_START", "HDFS:").rstrip())
print("\nMetastore:")
print(tables_part.rstrip())

missing = [t for t in ALL_TABLES if t not in tables_part]
assert not missing, (
    f"{SRC_DB} does not list {missing}. Re-run Steps 1-2: without the fixture, discovery emits a "
    f"single DATABASE_NOT_FOUND record for the '*' token, the run still reports success, and "
    f"Step 9 fails ten minutes later with a table set of ['*']."
)
for t in HEALTHY_TABLES:
    assert f"{HDFS_SRC_BASE}/{t}" in hdfs_part, f"{t}: data directory missing — re-run Step 1"
for t in ORPHAN_TABLES:
    assert f"{HDFS_SRC_BASE}/{t}" not in hdfs_part, \
        f"{t}: data directory is present, so it is not orphaned — re-run Step 2"

print(f"\nFixture intact: {len(ALL_TABLES)} tables registered, {len(ORPHAN_TABLES)} orphaned. "
      f"Triggering.\n")

logical_date = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S+00:00")
dag_run_name = f"manual_wf343_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

trigger_body = {
    "dag_run_id": dag_run_name,
    "logical_date": logical_date,
    "conf": {"excel_file_path": EXCEL_S3_PATH},
}

resp = airflow_request("POST", f"/api/v2/dags/{DAG_ID}/dagRuns", json=trigger_body)
print(f"Triggered: dag_run_id={resp.get('dag_run_id')} state={resp.get('state')}")
print(f"  excel_file_path = {EXCEL_S3_PATH}")
DAG_RUN_ID = resp["dag_run_id"]

---
## Step 8 — Monitor, then assert discovery survived

This is the WF-343 regression test. The DAG-run state is **not** the signal to assert on: tasks use
`trigger_rule='all_done'`, so the run can reach `success` with a failed discovery behind it. The
signal is the mapped `discover_tables_via_spark_ssh` instance itself.

Against `main` that instance fails with `Discovery failed for 2/4 table(s) in wf343_db`, and this is
the cell where the notebook stops.

In [ ]:
import time

TERMINAL = {"success", "failed"}
POLL_INTERVAL = 15
TIMEOUT_SECS = 60 * 30

deadline = time.time() + TIMEOUT_SECS
last_state = None
while time.time() < deadline:
    run = airflow_request("GET", f"/api/v2/dags/{DAG_ID}/dagRuns/{DAG_RUN_ID}")
    state = run.get("state")
    if state != last_state:
        print(f"[{time.strftime('%H:%M:%S')}] state = {state}")
        last_state = state
    if state in TERMINAL:
        break
    time.sleep(POLL_INTERVAL)
else:
    print(f"Timed out after {TIMEOUT_SECS}s waiting for terminal state.")

tasks = airflow_request("GET", f"/api/v2/dags/{DAG_ID}/dagRuns/{DAG_RUN_ID}/taskInstances")
instances = tasks.get("task_instances", [])

print("\nTask summary:")
print(f"  {'task_id':<38} {'map':<5} {'state':<10} duration")
print(f"  {'-'*38} {'-'*5} {'-'*10} {'-'*10}")
for t in sorted(instances, key=lambda x: (x.get("start_date") or "")):
    print(f"  {t.get('task_id',''):<38} {str(t.get('map_index','')):<5} "
          f"{str(t.get('state','')):<10} {t.get('duration')}")

In [ ]:
discovery = [t for t in instances if t.get("task_id") == "discover_tables_via_spark_ssh"]
assert discovery, "no discover_tables_via_spark_ssh instance found for this run"

states = {t.get("map_index"): t.get("state") for t in discovery}
print(f"discover_tables_via_spark_ssh states by map_index: {states}")

failed = [idx for idx, st in states.items() if st != "success"]
assert not failed, (
    f"Discovery did not survive the orphaned entries — map_index {failed} in state "
    f"{[states[i] for i in failed]}. This is the WF-343 regression: check the task log for "
    f"'Discovery failed for N/{len(ALL_TABLES)} table(s) in {SRC_DB}'."
)

print(f"\nPASS — discovery succeeded despite {len(ORPHAN_TABLES)} orphaned metastore entr(ies).")

---
## Step 9 — Per-table tracking assertions

The interesting columns are `overall_status`, `discovery_status` and `error_message` in
`{TRACKING_DB}.migration_table_status`. Reading them needs Ranger `select` on that database, and
`TRACKING_DB` must be the database the DAG actually wrote to — Step 4 pins it as a Variable so the
two cannot drift.

`legacy_events` gets its own assertion. Because a TEXTFILE orphan never raises, it used to emit as a
*success* record, reach DistCp with 0 source files, and be recorded `EMPTY_SOURCE` — which
`finalize_run` counts as **successful**. The condition this fix exists to surface was still passing
silently for every non-Parquet table.

In [ ]:
rid_rows = spark.sql(f"""
    SELECT run_id FROM {TRACKING_DB}.migration_runs
    WHERE dag_run_id = '{DAG_RUN_ID}'
    ORDER BY started_at DESC LIMIT 1
""").collect()
assert rid_rows, f"no migration_runs row for dag_run_id={DAG_RUN_ID}"
RUN_ID = rid_rows[0]["run_id"]
print(f"run_id = {RUN_ID}\n")

status_rows = spark.sql(f"""
    SELECT source_table, overall_status, discovery_status, distcp_status,
           validation_status, source_location, source_file_count, error_message
    FROM {TRACKING_DB}.migration_table_status
    WHERE run_id = '{RUN_ID}'
    ORDER BY source_table
""").collect()

by_table = {r["source_table"]: r for r in status_rows}

print(f"  {'table':<18} {'overall':<24} {'discovery':<24} {'distcp':<20} validation")
print(f"  {'-'*18} {'-'*24} {'-'*24} {'-'*20} {'-'*12}")
for r in status_rows:
    print(f"  {r['source_table']:<18} {str(r['overall_status']):<24} "
          f"{str(r['discovery_status']):<24} {str(r['distcp_status']):<20} "
          f"{r['validation_status']}")

# A lone '*' row means the token never resolved: discovery found no such database and emitted
# one DATABASE_NOT_FOUND record for the literal token. The run still reports success.
if set(by_table) == {"*"}:
    raise AssertionError(
        f"Discovery recorded a single row with source_table='*', so {SRC_DB} did not exist on the "
        f"edge node when this run started. Re-run Steps 1-2 to rebuild the fixture, then trigger "
        f"again — Step 6b now catches this before the DAG runs."
    )

assert set(by_table) == set(ALL_TABLES), \
    f"table set mismatch: {sorted(by_table)} != {ALL_TABLES}"

In [ ]:
# --- the orphans ---
for t in ORPHAN_TABLES:
    row = by_table[t]
    assert row["overall_status"] == "SOURCE_PATH_NOT_FOUND", \
        f"{t}: overall_status={row['overall_status']!r}, expected SOURCE_PATH_NOT_FOUND"
    assert row["discovery_status"] == "SOURCE_PATH_NOT_FOUND", \
        f"{t}: discovery_status={row['discovery_status']!r}"
    # The path is the actionable detail: it decides whether ops drops the metastore entry
    # or restores the data.
    assert t in (row["error_message"] or ""), \
        f"{t}: error_message does not name the table's path: {row['error_message']!r}"
    print(f"  PASS  {t:<18} SOURCE_PATH_NOT_FOUND")

# legacy_events is the silent-false-success case: EMPTY_SOURCE counts as successful.
assert by_table["legacy_events"]["overall_status"] != "EMPTY_SOURCE", (
    "legacy_events was recorded EMPTY_SOURCE — a TEXTFILE orphan is passing as a successful "
    "empty copy, which is the pre-fix behaviour for non-Parquet tables."
)

# --- the healthy siblings ---
for t in HEALTHY_TABLES:
    row = by_table[t]
    assert row["overall_status"] == "VALIDATED", \
        f"{t}: overall_status={row['overall_status']!r}, expected VALIDATED. " \
        f"error_message={row['error_message']!r}"
    print(f"  PASS  {t:<18} VALIDATED — {row['source_file_count']} source file(s)")

print(f"\nPASS — {len(ORPHAN_TABLES)} orphan(s) skipped and flagged, "
      f"{len(HEALTHY_TABLES)} sibling(s) migrated end to end.")

In [ ]:
# Which path caught each orphan? Both must reach the same verdict, and it is worth seeing
# which one fired: a Parquet orphan raises inside spark.table(), a TEXTFILE orphan never
# raises and is caught by the source_path_exists check instead.
CANNED = "Source data path does not exist:"
for t in ORPHAN_TABLES:
    msg = by_table[t]["error_message"] or ""
    mechanism = "never raised (caught by fs.exists)" if msg.startswith(CANNED) else "raised during discovery"
    print(f"  {t:<18} {mechanism}")
    print(f"  {'':<18} {msg[:140]}")

---
## Step 10 — Run-level status

A skipped table must never leave the run looking clean. `finalize_run` counts the skippable statuses
in `not_found`, and `not_found > 0` downgrades the run to `COMPLETED_WITH_MISSING`.

If this reports `COMPLETED_WITH_FAILURES` instead, a healthy sibling failed for an unrelated
environment reason — `failed > 0` takes precedence over `not_found > 0`. Step 9 will already have
named the table, so read that output first.

In [ ]:
run_row = spark.sql(f"""
    SELECT status, total_tables, successful_tables, failed_tables
    FROM {TRACKING_DB}.migration_runs WHERE run_id = '{RUN_ID}'
""").collect()[0]

print(f"  status            = {run_row['status']}")
print(f"  total_tables      = {run_row['total_tables']}")
print(f"  successful_tables = {run_row['successful_tables']}")
print(f"  failed_tables     = {run_row['failed_tables']}")

assert run_row["status"] != "COMPLETED", (
    "Run finished a bare COMPLETED with skipped tables in it — the skip is only acceptable "
    "because it downgrades the run status."
)
assert run_row["status"] == "COMPLETED_WITH_MISSING", (
    f"status={run_row['status']!r}, expected COMPLETED_WITH_MISSING. COMPLETED_WITH_FAILURES "
    f"means a sibling failed for an unrelated reason — see Step 9."
)

print("\nPASS — run downgraded to COMPLETED_WITH_MISSING.")

---
## Step 11 — The report makes it visible

An operator should not have to read task logs to find out a table went unmigrated. The report gains
a `SOURCE PATH MISSING` summary card and a distinct badge, and renders the missing location beside
the row (HTML-escaped) rather than the full multi-line Spark exception.

In [ ]:
from IPython.display import HTML, display

report_path = f"{REPORT_LOCATION}/{RUN_ID}_report.html"
print(f"Reading report: {report_path}\n")

fs = _fs(report_path)
p = spark._jvm.org.apache.hadoop.fs.Path(report_path)
assert fs.exists(p), (
    f"report not found at {report_path} — REPORT_LOCATION may differ from the DAG's "
    f"migration_report_location Variable."
)

reader = spark._jvm.java.io.BufferedReader(
    spark._jvm.java.io.InputStreamReader(fs.open(p), "UTF-8")
)
lines = []
line = reader.readLine()
while line is not None:
    lines.append(line)
    line = reader.readLine()
reader.close()
report_html = "\n".join(lines)

assert "SOURCE PATH MISSING" in report_html, "report is missing the SOURCE PATH MISSING card"
assert "status-path-not-found" in report_html, "report is missing the SOURCE_PATH_NOT_FOUND badge class"
for t in ORPHAN_TABLES:
    assert t in report_html, f"{t} does not appear in the report"

print(f"PASS — card, badge and both orphan rows present ({len(report_html)} bytes).")
display(HTML(report_html))

---
## Step 12 — Cleanup

Removes everything this notebook created: the source database and its HDFS data on the edge node,
the migrated S3 destination and its Hive database, the Excel config, the report HTML, and this run's
tracking rows.

Leaves the Airflow Variables and the `maprlogin` stub in place — they are shared setup that the
DAG-1 harness also expects. The destination Hive database is likely to survive too, for the same
Ranger reason as Step 6; its S3 data is still removed.

In [ ]:
print("Cleaning up test artifacts...\n")

CLEANUP_SCRIPT = r"""# -*- coding: utf-8 -*-
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("wf343-cleanup").enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel("WARN")
spark.sql("DROP DATABASE IF EXISTS __SRC_DB__ CASCADE")
print("dropped database __SRC_DB__")
""".replace("__SRC_DB__", SRC_DB)

try:
    edge_pyspark(CLEANUP_SCRIPT, "cleanup")
    pod_exec(f"hdfs dfs -rm -r -skipTrash {HDFS_SRC_BASE} 2>/dev/null || true", quiet=True)
    print(f"  removed HDFS {HDFS_SRC_BASE}")
except Exception as e:
    print(f"  edge cleanup skipped: {e}")

for path in (f"{TENANT_PREFIX}/{DEST_DB}", EXCEL_S3_PATH, f"{REPORT_LOCATION}/{RUN_ID}_report.html"):
    try:
        s3_delete(path)
    except Exception as e:
        print(f"  ERROR deleting {path}: {e}")

try_drop_database(DEST_DB, f"{DEST_DB} will be left behind")

try:
    spark.sql(f"DELETE FROM {TRACKING_DB}.migration_table_status WHERE run_id = '{RUN_ID}'")
    spark.sql(f"DELETE FROM {TRACKING_DB}.migration_runs        WHERE run_id = '{RUN_ID}'")
    print(f"  deleted tracking rows for run_id={RUN_ID}")
except Exception as e:
    print(f"  tracking cleanup skipped: {e}")

print("\nCleanup complete.")